# 01.01 · Chat models and `init_chat_model`

**Module 01 · Models, Messages & Prompts** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 90–120 min |
| 📋 Prerequisites | Module 00 complete (course env `lc-mastery`, `mastery` installed editable, `.env` with `GROQ_API_KEY`). |
| 🔑 Keys | `GROQ_API_KEY` (optional: without it, 💸 cells are skipped) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-11 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (fake model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

By the end of this lesson you can:

1. **Explain** why every LangChain chat model shares one interface (`BaseChatModel`, a `Runnable`) and what a provider integration package adds.
2. **Create** the same model three ways: `ChatGroq(...)`, `init_chat_model("groq:...")`, and the course helper `mastery.models.get_model(role)`, and choose between them.
3. **Configure** temperature, max tokens, timeouts and retries, and **find the real defaults** in the code when the docs disagree.
4. **Call** a model with `invoke`, `stream` and `batch`, and pass a `RunnableConfig` (`run_name`, `tags`, `metadata`, `max_concurrency`).
5. **Use** the Jupyter magics `%pip`, `%env` and `%%time`, and read a model's **profile** (capabilities) before you depend on it.

## 📚 Official reading (verified 2026-10-11)

| Page | Read it for |
|---|---|
| [Models](https://docs.langchain.com/oss/python/langchain/models) | **Main page.** Initialize, parameters, connection resilience, invoke/stream/batch, profiles, exceptions, invocation config |
| [Providers and models](https://docs.langchain.com/oss/python/concepts/providers-and-models) | How providers, integration packages and model names fit together |
| [Chat model integrations](https://docs.langchain.com/oss/python/integrations/chat/index) | Which providers exist and which features each supports |
| [`ChatGroq` reference](https://reference.langchain.com/python/langchain-groq/chat_models/ChatGroq) · [ChatGroq integration page](https://docs.langchain.com/oss/python/integrations/chat/groq) | Every Groq-specific parameter |
| [`init_chat_model` reference](https://reference.langchain.com/python/langchain/chat_models/base/init_chat_model) | Provider prefixes, `configurable_fields` |
| [`RunnableConfig` reference](https://reference.langchain.com/python/langchain-core/runnables/config/RunnableConfig) | All config keys |
| [IPython built-in magics](https://ipython.readthedocs.io/en/stable/interactive/magics.html) | `%pip`, `%env`, `%%time` and friends |

> **Read the Models page twice:** once now (skim the headings), once after this lesson. You'll notice details the second time that only make sense after running the code.

## 🧰 This module's Jupyter skill: magics and rich display

**Magics** are IPython commands that start with `%` (line magic: applies to one line) or `%%` (cell magic: applies to the whole cell and **must be the first line** of the cell). They are not Python; the kernel intercepts them. You already used `%pip` in Module 00. This module adds:

| Magic | What it does | Where you'll use it |
|---|---|---|
| `%pip install ...` | Installs into **this kernel's** environment | Every lesson's Step 0 |
| `%env NAME=value` / `%env NAME` | Sets / reads an environment variable **for this kernel process only** | Part 3 below |
| `%%time` | Times the whole cell (wall time vs CPU time) | Parts 6–7, Lesson 01.05 |
| `%lsmagic` · `%magic_name?` | Lists all magics · shows help for one | Try them now |

**Rich display** means showing objects as HTML, Markdown, tables or images instead of plain text: `IPython.display.display(...)`, `Markdown(...)`, `HTML(...)`, and pandas tables. You'll build up to a live-updating streaming display in Lesson 01.05.

In [ ]:
# 🧪 Explore magics. %lsmagic is itself a magic: the kernel answers, not Python.
%lsmagic

**You should now see** two lists: *Available line magics* (`%alias`, `%autoreload`, `%env`, `%pip`, `%time`, ...) and *Available cell magics* (`%%bash`, `%%html`, `%%time`, `%%writefile`, ...). Type `%env?` in a new cell to read its help.

---
## Step 0 · Install this lesson's packages

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "pandas>=3.0.6,<4"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-core", "langchain-groq", "groq", "pandas",):
    print(f"{package:<22} {metadata.version(package)}")

**You should now see** `langchain 1.4.x`, `langchain-core 1.6.x`, `langchain-groq 1.1.x`, `groq 0.3x.x` and `pandas 3.x` (or newer patch/minor versions).

> **Why is `groq` at 0.x when PyPI shows 1.x?** `langchain-groq` 1.1.3 requires `groq<1.0.0` (verified 2026-10-11 on PyPI). pip respects that, so you get the newest 0.x. This is normal dependency resolution, and the reason the course pins ranges instead of "latest of everything".

## Step 1 · Setup

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/01-models-messages-prompts/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

**You should now see** the repo root, the `mastery` version and the file it was loaded from (inside this repo's `src/`; a ⚠️ line means your environment's install needs the one-line fix it prints), `GROQ_API_KEY : set: gsk_…` (masked, never the full key) or `not set`, and the two model IDs from your `.env` (`openai/gpt-oss-20b` and `openai/gpt-oss-120b` unless you changed them).

---
## Part 1 · One interface, many providers

LangChain has **one abstract class** for chat models, `BaseChatModel` (in `langchain-core`), and **one integration package per provider** (`langchain-groq`, `langchain-openai`, `langchain-google-genai`, ...). Each package implements the same methods by translating to its provider's API. That's why you can switch providers without rewriting your application ([Providers and models](https://docs.langchain.com/oss/python/concepts/providers-and-models)).

`BaseChatModel` is itself a **`Runnable`**: the protocol (`invoke`, `stream`, `batch` and their `a...` async twins) that prompts, tools, agents and LangGraph graphs also implement. Lesson 01.07 opens that box; for now, look at the family tree.

In [ ]:
# 🧪 Offline: inspect the class hierarchy. No network, no key needed.
from langchain_groq import ChatGroq

for cls in ChatGroq.__mro__:
    if cls.__module__.startswith(("langchain", "pydantic")) and "[" not in cls.__name__:
        print(f"{cls.__module__}.{cls.__name__}")

In [ ]:
# 🧪 The standard methods every chat model has, whatever the provider.
STANDARD_METHODS = ["invoke", "stream", "batch", "batch_as_completed",
                    "ainvoke", "astream", "abatch", "with_structured_output",
                    "bind_tools", "with_retry", "with_fallbacks", "with_config"]
print({name: hasattr(ChatGroq, name) for name in STANDARD_METHODS})

### What just happened?

- `ChatGroq` → `BaseChatModel` → `BaseLanguageModel` → `RunnableSerializable` → `Runnable`. Everything after `ChatGroq` lives in `langchain_core`, shared by all providers.
- All twelve methods are `True`. You'll meet each one in this module (structured output in 01.04, async in 01.05, retries/fallbacks in 01.06–01.07). `bind_tools` arrives with agents in Module 02.

---
## Part 2 · Three ways to create the same model

| Way | Code | Use it when |
|---|---|---|
| **Provider class** | `ChatGroq(model="openai/gpt-oss-20b", ...)` | You need a provider-specific option with IDE autocompletion, or you're reading provider docs |
| **`init_chat_model`** | `init_chat_model("groq:openai/gpt-oss-20b", ...)` | Provider-agnostic code: the provider is just part of a string you can keep in config |
| **Course helper** | `get_model("fast")` | In this course: the ID comes from `.env` (`MASTERY_MODEL_FAST`), you get friendly errors, and preview models warn you |

`get_model` is a thin wrapper around `init_chat_model` (read it: [`src/mastery/models.py`](../../src/mastery/models.py)). The `"groq:..."` prefix tells `init_chat_model` which integration package to import. Only the **first** colon splits provider from model, which matters because Groq model IDs contain a slash.

**Construction is free.** Creating the object does not call Groq; the first network request happens on `invoke`. So we can inspect models offline with a placeholder key.

In [ ]:
# 🧪 Offline: build the fast model three ways. OFFLINE_KEY is a placeholder: no request is sent.
from langchain.chat_models import init_chat_model

from mastery.models import describe_model, get_model

OFFLINE_KEY = "gsk_OFFLINE_INSPECTION_ONLY"   # Not a real key; construction never calls the API.

by_class = ChatGroq(model=settings.model_fast, api_key=OFFLINE_KEY)
by_init = init_chat_model(f"groq:{settings.model_fast}", api_key=OFFLINE_KEY)
by_helper = get_model("fast", check_key=False, api_key=OFFLINE_KEY)

for label, model in (("ChatGroq(...)", by_class), ("init_chat_model", by_init), ("get_model", by_helper)):
    info = describe_model(model)
    print(f"{label:<16} → {info['class']}  model={info['model']}  max_retries={info['max_retries']}  timeout={info['timeout']}")

### What just happened?

- All three produced a `ChatGroq` for the same model ID. `init_chat_model` is not a different kind of model; it's a **factory** that picks the class for you.
- `max_retries=2` for the first two, because that is **`ChatGroq`'s default**. `get_model` also sets `timeout=60`, so a stuck request can't hang your notebook forever (the class default is `None` = wait as long as the HTTP client allows).
- In your own code outside the course, use `init_chat_model` with the model string in config. Use the provider class when you need its specific features.

---
## Part 3 · `%env`: change configuration without editing files

`get_model("fast")` reads `MASTERY_MODEL_FAST` through `load_settings()`. Real environment variables **win over** `.env` (twelve-factor rule, Lesson 00.04). `%env NAME=value` sets a variable **in this kernel's process only**: no file changes, nothing persists after a restart. That makes it a safe way to try another model.

In [ ]:
# 🧪 Remember the current value so we can restore it afterwards.
import os

original_fast = os.environ.get("MASTERY_MODEL_FAST")
print("Before:", get_model("fast", check_key=False, api_key=OFFLINE_KEY).model_name)

In [ ]:
%env MASTERY_MODEL_FAST=openai/gpt-oss-120b

In [ ]:
# 🧪 Same call, different model: configuration changed, code did not.
print("After %env:", get_model("fast", check_key=False, api_key=OFFLINE_KEY).model_name)

# Restore (plain Python works too: %env is a convenience, os.environ is the real thing).
if original_fast is None:
    os.environ.pop("MASTERY_MODEL_FAST", None)
else:
    os.environ["MASTERY_MODEL_FAST"] = original_fast
print("Restored:", get_model("fast", check_key=False, api_key=OFFLINE_KEY).model_name)

**You should now see** `Before: openai/gpt-oss-20b`, then `env: MASTERY_MODEL_FAST=openai/gpt-oss-120b`, then `After %env: openai/gpt-oss-120b`, then `Restored: openai/gpt-oss-20b`.

> 🔐 **`%env` and secrets.** Bare `%env` (no arguments) lists every variable but hides names containing `key`, `token` or `secret`. **`%env GROQ_API_KEY` prints the full key**, and that output gets saved in the notebook. Never run `%env` on a secret name; use `mastery.config.env_status(...)` (masked) instead. `%env NAME` also raises `UsageError` if the variable doesn't exist, which would break *Run All*; that's why this lesson sets a value before reading it.

---
## Part 4 · Parameters, and where the real defaults live

The [Models page](https://docs.langchain.com/oss/python/langchain/models#parameters) lists the **standard** parameters every integration accepts:

| Parameter | Meaning | Course guidance |
|---|---|---|
| `model` | Model ID (or `"provider:model"` with `init_chat_model`) | From config, never hard-coded |
| `temperature` | Randomness: low = repeatable, high = varied | 0 for extraction/classification; see the box below for reasoning models |
| `max_tokens` | Cap on **output** tokens | Set it when you need short answers or a cost ceiling |
| `timeout` | Seconds to wait for a response | Always set one in notebooks and services |
| `max_retries` | Automatic retries with exponential backoff for 429, 5xx and network errors | Keep the default; set 0 only in failure demos |
| `rate_limiter` | Client-side throttle | Lesson 01.06 |

Provider-specific extras (for Groq: `reasoning_effort`, `reasoning_format`, `service_tier`, ...) are listed on the [`ChatGroq` reference](https://reference.langchain.com/python/langchain-groq/chat_models/ChatGroq).

**The docs describe the concept; the code decides the default.** The Models page says `max_retries` defaults to 6. Let's ask the code.

In [ ]:
# 🧪 Read the declared defaults straight from the Pydantic fields of ChatGroq.
import pandas as pd
from IPython.display import display

FIELDS = ["temperature", "max_tokens", "max_retries", "request_timeout", "reasoning_effort", "reasoning_format"]
defaults = pd.DataFrame(
    [{"parameter": name, "ChatGroq default": repr(ChatGroq.model_fields[name].default)} for name in FIELDS]
)
display(defaults)

In [ ]:
# 🧪 A quirk you only find in the source: ChatGroq replaces temperature=0 with 1e-8.
print(ChatGroq(model=settings.model_fast, api_key=OFFLINE_KEY, temperature=0).temperature)

### What just happened?

- `ChatGroq` defaults: `temperature=0.7`, `max_retries=2`, `request_timeout=None`, `max_tokens=None`. The **6** on the Models page is a generic description; each integration sets its own default. When it matters (cost, latency, retries), **set the parameter explicitly** instead of trusting a default you haven't read.
- `temperature=0` became `1e-08`. `langchain-groq` does this on purpose (see `validate_environment` in its `chat_models.py`); for you it behaves like 0. You found it by reading code, the most reliable docs there are (Lesson 00.06).

> 🌡️ **The docs disagree, so you decide with data.** Groq's [Prompt basics](https://console.groq.com/docs/prompting) recommends temperature **0.0 for data extraction**; Groq's [Reasoning page](https://console.groq.com/docs/reasoning) recommends **0.5–0.7 for reasoning models** "to prevent repetitions or incoherent outputs", and `gpt-oss` models *are* reasoning models. Both can be right for different tasks. This course uses **0** for extraction and classification (repeatable, easy to evaluate), and the default otherwise. Project 01 gives you the numbers to check this choice for your task.

---
## Part 5 · Model profiles: what can this model do?

Since `langchain` 1.1, chat models expose a **`profile`** dict: context window, image input, tool calling, structured output, reasoning, and more ([Models → Model profiles](https://docs.langchain.com/oss/python/langchain/models#model-profiles)). The data comes from the open [models.dev](https://models.dev/) project, **shipped inside the integration package**. Agents use it later (for example, summarization middleware reads the context window).

In [ ]:
# 🧪 Compare the three course roles. The vision model is a PREVIEW model: watch for the warning.
rows = []
for role in ("fast", "reasoning", "vision"):
    info = describe_model(get_model(role, check_key=False, api_key=OFFLINE_KEY))
    rows.append({"role": role} | info)
display(pd.DataFrame(rows, dtype=object).drop(columns=["temperature", "max_tokens", "timeout", "max_retries"]))

### What just happened?

- A **`PreviewModelWarning`** appeared for `qwen/qwen3.8-27b`: `get_model` reminds you every time that preview models are for evaluation only.
- The two `gpt-oss` rows show `max_input_tokens=131072`, `structured_output=True`, `reasoning_output=True`, `image_inputs=False`.
- The vision row is **all `None`**. The profile data in `langchain-groq` 1.1.3 (released 2026-06-10) predates `qwen/qwen3.8-27b`, so LangChain doesn't know it. The model *does* accept images ([Groq vision docs](https://console.groq.com/docs/vision)). **Profiles are a beta feature and can lag behind new models.** If you need one, pass your own: `init_chat_model(..., profile={"image_inputs": True, "max_input_tokens": 131072})`.

---
## Part 6 · `invoke`, `stream`, `batch`

| Method | Returns | Use it for |
|---|---|---|
| `invoke(input)` | One `AIMessage` when the answer is complete | Scripts, extraction, anything that needs the whole answer |
| `stream(input)` | An iterator of `AIMessageChunk`s as tokens arrive | Chat UIs: users see progress immediately |
| `batch([input, ...])` | A list of `AIMessage`s, in input order | Many independent requests in parallel (client-side threads) |

`input` can be a string (shortcut for one user message), a list of messages, or a list of dicts (Lesson 01.02). First, the **shapes**, offline, with LangChain's test double `GenericFakeChatModel` ([unit testing docs](https://docs.langchain.com/oss/python/langchain/test/unit-testing)). It replays scripted answers, so you can see exactly what each method returns without a key.

In [ ]:
# 🧪 Offline: the same three methods on a fake model that replays scripted answers.
from langchain.messages import AIMessage
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel

fake = GenericFakeChatModel(messages=iter([AIMessage("Parrots are colorful to attract mates."),
                                           AIMessage("Streaming shows tokens as they arrive.")]))
reply = fake.invoke("Why are parrots colorful?")
print(type(reply).__name__, "→", reply.text)

for chunk in fake.stream("What does streaming do?"):
    print(f"[{chunk.text}]", end="")   # Each chunk is an AIMessageChunk; brackets show the pieces.

**You should now see** `AIMessage → Parrots are colorful to attract mates.` and then the second answer cut into pieces: `[Streaming][ ][shows][ ]...`. Real models stream in tokens; the fake splits on spaces. Now the real thing.

In [ ]:
%%time
# 💸 Calls Groq (fast model), ~150 tokens. %%time must be the FIRST line of the cell.
from mastery.display import show_message

if not HAVE_GROQ_KEY:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")
else:
    llm = get_model("fast", temperature=0, max_tokens=300, reasoning_effort="low")
    response = llm.invoke("In two sentences: what is a chat model in LangChain?")
    show_message(response)   # Rich display: an HTML card (Lesson 01.02 explains every field).

**You should now see** an HTML card with the answer, a collapsed **reasoning** section (gpt-oss models think before answering), and a token line. Below it, `%%time` prints `CPU times: ...` and **`Wall time: ...`**. Wall time (≈0.5–2 s on Groq) is what the user waits; CPU time is tiny because your computer mostly waits for the network.

In [ ]:
# 💸 Calls Groq (fast model), ~150 tokens. Prints tokens as they arrive.
if not HAVE_GROQ_KEY:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")
else:
    for chunk in llm.stream("List three uses of streaming in one short line each."):
        print(chunk.text, end="", flush=True)   # flush=True: show each piece immediately.

In [ ]:
%%time
# 💸 Calls Groq 3 times in parallel (fast model), ~400 tokens in total.
QUESTIONS = ["What is a token?", "What is a context window?", "What is temperature?"]

if not HAVE_GROQ_KEY:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")
else:
    answers = llm.batch([q + " One sentence." for q in QUESTIONS], config={"max_concurrency": 3})
    for question, answer in zip(QUESTIONS, answers):
        print(f"Q: {question}\nA: {answer.text}\n")

### What just happened?

- `invoke` waited for the full answer; `stream` printed it progressively; `batch` sent three requests **at the same time** (threads) and returned answers **in input order**. Its wall time is close to one call, not three.
- `max_concurrency` caps how many requests run at once. On the free tier, concurrency is how you hit rate limits fast (Lesson 01.06).
- `batch()` is **client-side parallelism**, not a provider "Batch API" (an asynchronous, discounted job service). The [Models page](https://docs.langchain.com/oss/python/langchain/models#batch) points out the difference.

---
## Part 7 · `RunnableConfig`: run-time settings for one call

Every `invoke`/`stream`/`batch` accepts `config=`, a dict ([`RunnableConfig`](https://reference.langchain.com/python/langchain-core/runnables/config/RunnableConfig)). The keys you'll use most:

| Key | Effect |
|---|---|
| `run_name` | Name of this run in traces (Lesson 01.08). Not inherited by sub-calls |
| `tags` | List of labels, inherited by every sub-call (filter traces by them) |
| `metadata` | Dict of key/values, inherited (user ID, experiment, prompt version...) |
| `max_concurrency` | Parallelism limit for `batch` |
| `callbacks` | Objects notified on events (start, new token, end, error) |

A **callback handler** lets you watch config flow through a call, offline.

In [ ]:
# 🧪 Offline: a tiny callback handler that prints what the model receives in its config.
from langchain_core.callbacks import BaseCallbackHandler


class ConfigSpy(BaseCallbackHandler):
    """Print the run name, tags and metadata each time a chat model starts."""

    def on_chat_model_start(self, serialized, messages, *, run_id, tags=None, metadata=None, **kwargs):
        print("name    :", kwargs.get("name"))
        print("tags    :", tags)
        print("metadata:", {k: v for k, v in (metadata or {}).items() if not k.startswith(("ls_", "lc_"))})


fake = GenericFakeChatModel(messages=iter([AIMessage("ok")]))
_ = fake.invoke("hi", config={"run_name": "lesson-01-01-demo", "tags": ["m01", "demo"],
                          "metadata": {"lesson": "01.01", "user_id": "learner-1"}, "callbacks": [ConfigSpy()]})

### What just happened?

The handler received `name: lesson-01-01-demo`, your tags and your metadata. (We hid keys starting with `ls_` and `lc_`: LangChain adds those itself, e.g. `ls_provider` and `lc_versions`, for LangSmith.) Lesson 01.08 sends exactly this information to LangSmith, where tags and metadata become filters.

---
## Part 8 · Connection resilience: what is retried, what is not

From the [Models page](https://docs.langchain.com/oss/python/langchain/models#connection-resilience) (verified 2026-10-11): chat models retry **network errors, rate limits (429) and server errors (5xx)** with exponential backoff and jitter. **Client errors like 401 (bad key) or 404 are not retried**, because retrying cannot fix them.

Let's prove it with a fake key. If 401s were retried, this would take several seconds; it fails at once.

In [ ]:
# 🌐 Sends ONE request with a fake key (rejected with 401, no tokens). Needs internet, no key.
import time

from mastery.models import explain_model_error

bad = ChatGroq(model=settings.model_fast, api_key="gsk_FAKE_key_for_retry_demo", max_retries=2, timeout=20)
start = time.perf_counter()
try:
    bad.invoke("hello")
except Exception as error:   # Broad on purpose: we classify it below.
    info = explain_model_error(error)
    print(f"{type(error).__module__}.{type(error).__name__} after {time.perf_counter() - start:.2f}s")
    print(f"kind={info.kind}  retryable={info.retryable}  status={info.status_code}")
    print("hint:", info.hint)

In [ ]:
# 🧪 Which exception hierarchy does each integration use? (Checked offline with isinstance.)
import groq
from langchain_core.exceptions import ModelAuthenticationError, ModelError

print("groq.AuthenticationError is a LangChain ModelError?",
      issubclass(groq.AuthenticationError, ModelError))
print("LangChain standard auth error:", ModelAuthenticationError.__module__ + "." + ModelAuthenticationError.__name__)

### What just happened?

- The 401 came back in well under a second (no internet? you'll see `kind=connection` instead, after the retries; that's the retryable case).
- The error was **`groq.AuthenticationError`**, the Groq SDK's class, and it is **not** a LangChain `ModelError`. The Models page says *major* integrations raise standard exceptions from `langchain_core.exceptions` (`ModelAuthenticationError`, `ModelRateLimitError`, ...). On 2026-10-11, `langchain-openai` 1.7.0 does; `langchain-groq` 1.1.3 does not yet.
- That's why `mastery.models.explain_model_error` handles **both**: standard types first, then the HTTP `status_code` that SDK errors carry. Write error handling against behavior you verified, not against the newest docs.

---
## Part 9 · `get_model` in practice

The helper turns every common mistake into a clear message. The paid benchmark (`provider="benchmark"`, OpenAI by default) needs `OPENAI_API_KEY` and `langchain-openai`; most learners have neither yet, which is exactly the case to handle well.

In [ ]:
# 🧪 Offline: what happens when a key or a package is missing? No crash, a clear message.
from mastery.config import MissingSettingError

try:
    benchmark = get_model("fast", provider="benchmark")
    print("Benchmark model ready:", describe_model(benchmark)["model"])
except MissingSettingError as error:
    print("MissingSettingError →", str(error).splitlines()[0])
except ImportError as error:
    print("ImportError →", error)

**You should now see** `MissingSettingError → ... OPENAI_API_KEY ...` (no key), or `ImportError → ... %pip install "langchain-openai" ...` (key set, package missing), or `Benchmark model ready: gpt-6-luna` if you have both.

> **Configurable models (read, don't memorize).** `init_chat_model(configurable_fields=("model", "temperature"))` creates a model whose ID is chosen **per call**: `model.invoke(..., config={"configurable": {"model": "groq:openai/gpt-oss-120b"}})`. Useful for A/B tests and model pickers; see [Models → Configurable models](https://docs.langchain.com/oss/python/langchain/models#configurable-models). In agents, middleware does this more cleanly (Lesson 04.02).

---
## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| `ValueError: Unable to infer model provider` | Model ID without `"provider:"` prefix in `init_chat_model` | Use `"groq:openai/gpt-oss-20b"` or pass `model_provider="groq"` |
| `UsageError: Environment does not have key: ...` | `%env NAME` on a variable that isn't set | Set it first, or use `os.environ.get(...)` |
| `%%time` → `UsageError: Line magic function %%time not found` | `%%time` was not the first line of the cell | Move it to line 1 (above comments) |
| A cell hangs for minutes | No `timeout`, slow network | `get_model(..., timeout=60)` (the helper's default) |
| Different answers on each run | `temperature` default is 0.7 for `ChatGroq` | Set `temperature=0` where repeatability matters |
| `profile` values are `None` | Model newer than the package's profile data | Pass `profile={...}` yourself; upgrade the integration later |
| `MissingSettingError` for `GROQ_API_KEY` | `.env` not loaded or key missing | Lesson 00.04; restart the kernel after editing `.env` |

## ⚠️ Production notes

> ⚠️ **Production note: trusting integration defaults.**
> This lesson shows that `ChatGroq` defaults (temperature 0.7, 2 retries, no timeout) differ from the generic docs. In production: set `timeout`, `max_retries` and `temperature` explicitly in one config module, log them with each request (LangSmith metadata), and re-check defaults when you upgrade an integration package.

> ⚠️ **Production note: preview models.**
> `get_model("vision")` returns a Groq **preview** model. In production: use a GA model with an SLA for anything users depend on, keep a fallback (Lesson 04.02), and watch [Groq deprecations](https://console.groq.com/docs/deprecations). Preview models are fine for experiments and for this course.

## 🏋️ Exercises

1. **(Easy)** Run `%env MASTERY_DEMO=hello`, then read it back with `%env MASTERY_DEMO` and with `os.environ["MASTERY_DEMO"]`. Restart the kernel and read it again. What happened, and why?
2. **(Easy)** Use `%%time` to compare `get_model("fast")` and `get_model("reasoning")` on the same question. Note wall time and `usage_metadata["output_tokens"]`.
3. **(Medium)** Without looking at the docs, find `ChatGroq`'s defaults for `n`, `streaming` and `service_tier` from `ChatGroq.model_fields`. Then find `service_tier` on the [`ChatGroq` reference](https://reference.langchain.com/python/langchain-groq/chat_models/ChatGroq) and write one sentence on what it does.
4. **(Stretch)** Write `compare_defaults(*classes)` that returns a DataFrame of `temperature`, `max_retries` and timeout defaults for several chat model classes. Try `ChatGroq` and, if installed, `ChatOpenAI` (`%pip install "langchain-openai>=1.7.0,<2"`). Hint: field names differ between classes; use `getattr(cls.model_fields.get(name), "default", "n/a")`.

<details>
<summary><b>Show solutions</b></summary>

**1.** `%env` changes this kernel's process environment, so both reads show `hello`. After a restart the kernel is a new process; the variable is gone (`UsageError` / `KeyError`). Persistent configuration belongs in `.env`.

**3.**
```python
for name in ("n", "streaming", "service_tier"):
    print(name, "=", repr(ChatGroq.model_fields[name].default))
```
Expect `n = 1`, `streaming = False`, `service_tier = 'on_demand'`. `service_tier` selects Groq's processing tier: `on_demand` (default), `auto`, or `flex`. Per [Groq's Flex Processing page](https://console.groq.com/docs/flex-processing) (verified 2026-10-11), `flex` is for paid plans only: 10x higher rate limits at the same price, but requests fail fast with HTTP **498** `capacity_exceeded` when capacity runs out, so you need jittered retries. (`explain_model_error` classifies 498 as a retryable server-side error.) Bonus research finding: that page's example still uses `llama-3.3-70b-versatile`, which Lesson 00.06 showed is retired.

**4.**
```python
def compare_defaults(*classes) -> pd.DataFrame:
    """Return the declared defaults of a few common parameters per chat model class."""
    rows = []
    for cls in classes:
        row = {"class": cls.__name__}
        for name in ("temperature", "max_retries", "request_timeout", "timeout"):
            field = cls.model_fields.get(name)
            row[name] = repr(field.default) if field is not None else "n/a"
        rows.append(row)
    return pd.DataFrame(rows)

display(compare_defaults(ChatGroq))
```
You'll find that `ChatOpenAI.max_retries` defaults to `None` (the OpenAI SDK's own default then applies). Same parameter name, different behavior: read the code.
</details>

## ✅ Checkpoint

- [ ] I can explain why `ChatGroq` and a future `ChatGoogleGenerativeAI` have the same `invoke`/`stream`/`batch`.
- [ ] I created the fast model three ways and know when to use each.
- [ ] I found `ChatGroq`'s real defaults in code (temperature 0.7, max_retries 2, timeout None).
- [ ] I used `%env` without ever printing a secret, and `%%time` on line 1 of a cell.
- [ ] (With a key) `invoke`, `stream` and `batch` returned answers; the 401 demo failed fast and was classified as `auth`.
- [ ] **Restart Kernel and Run All Cells** completes; then **Edit → Clear Outputs of All Cells** and save.

## 🔎 Docs drill (10 min)

Run `ChatGroq.validate_environment??`. Where do `max_retries` and `timeout` reach the `groq.Groq` client, and what does the validator do with `temperature=0`?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. In a cell: `ChatGroq.validate_environment??` (the Pydantic validator is a plain function, so `??` works).
2. It builds the client arguments (api key, base url, `timeout`, `max_retries`, headers) and creates both `groq.Groq` and `groq.AsyncGroq` with them.
3. Near the top, a `temperature` of exactly 0 is replaced with `1e-8`.
4. Bonus (GitHub): use the file's **Blame** view on that line to find the PR that added the substitution and why.

</details>

## 🔎 Research checkpoint (started here, logged in 01.02)

The module's question: *which providers support standard `content_blocks` natively, and is `langchain-groq` one of them?* Lesson 01.02 gives you the tools; note now which integration packages you have installed (`%pip list | grep langchain` in a cell, or `python -m pip list` in Git Bash).

## Summary

- One interface (`BaseChatModel`, a `Runnable`), many integration packages. `init_chat_model("provider:model")` is the provider-agnostic factory; `get_model(role)` adds course config and friendly errors.
- Construction is free; invocation costs. Inspect models offline, call them only when needed.
- Docs describe concepts; **code decides defaults**. Set `timeout`, `max_retries` and `temperature` explicitly.
- Profiles describe capabilities but can lag behind new models.
- Magics: `%pip` (install into this kernel), `%env` (process-only config; never on secrets), `%%time` (line 1 only).

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 01.02 · Messages and standard content blocks** → [`01-02-messages-and-content-blocks.ipynb`](01-02-messages-and-content-blocks.ipynb).

[Module 01 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)